# 2단계 파싱·정규화·피처 생성

커밋된 원본 텍스트를 읽어 중간 파싱 산출물을 다시 만들고, 스키마가 완성된 `data/processed/tidy_rewards.csv`를 저장한다. `data/raw/`는 읽기 전용으로만 사용한다.


In [1]:
from pathlib import Path
import re
import sys

import pandas as pd

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

raw_dir = project_root / 'data' / 'raw'
interim_dir = project_root / 'data' / 'interim'
processed_dir = project_root / 'data' / 'processed'
interim_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)

TODAY = pd.Timestamp('2026-06-10')


In [2]:
from src.features import compute_cycle_days, merge_is_discontinued
from src.normalize import categorize, extract_magnitude, normalize_reward
from src.parse import parse_by_type, parse_by_date, parse_date_header

by_type_text = (raw_dir / 'view_by_type.txt').read_text(encoding='utf-8')
by_date_text = (raw_dir / 'view_by_date.txt').read_text(encoding='utf-8')

by_type_records = parse_by_type(by_type_text)
by_date_records = parse_by_date(by_date_text)

parsed_by_type = pd.DataFrame(by_type_records, columns=[
    'reward_raw', 'date', 'stated_cycle_days', 'is_discontinued', 'trailing'
])
parsed_by_date = pd.DataFrame(by_date_records, columns=[
    'date', 'event_context', 'is_major_event', 'no_event_week', 'reward_raw', 'reward_trailing'
])


In [3]:
def is_special(row):
    text = ' '.join(
        str(x) for x in [row.get('reward_raw'), row.get('reward_trailing'), row.get('event_context')]
        if pd.notna(x)
    )
    return '\uD0C0\uC784' in text

def time_limited(trailing):
    if pd.isna(trailing) or trailing is None:
        return None
    text = str(trailing)
    if '\uD558\uB8E8 \uC885\uC77C' in text or '\uD558\uB8E8\uC885\uC77C' in text:
        return '\uD558\uB8E8 \uC885\uC77C'
    if '\uC624\uC804' in text or '\uC624\uD6C4' in text or '~' in text:
        return text
    if re.search(r'\d+\uBD84', text):
        return text
    return None

tidy_prenorm = parsed_by_date.copy()
tidy_prenorm['is_special'] = tidy_prenorm.apply(is_special, axis=1)
tidy_prenorm['time_limited'] = tidy_prenorm['reward_trailing'].apply(time_limited)
tidy_prenorm['is_future'] = pd.to_datetime(tidy_prenorm['date']) > TODAY
tidy_prenorm = tidy_prenorm[[
    'date', 'reward_raw', 'reward_trailing', 'event_context', 'is_major_event',
    'is_special', 'time_limited', 'is_future', 'no_event_week'
]]


## 7단계 스키마 결정

- 관측 단위는 `(date, reward_raw)`이고, `reward_norm`은 분석용 라벨이다.
- `(date, reward_norm)` 중복은 같은 날 함께 지급된 보상이 정규화로 묶인 경우이므로 제거하지 않고 보고만 한다.
- 미래 빈 블록은 삭제하지 않고 `is_future`로 구분한다.
- `cycle_days`는 고유 `(date, reward_norm)` 기준으로 계산한 뒤 원본 행에 다시 붙인다.
- `is_discontinued`는 `reward_raw` 본문 또는 `reward_raw + reward_trailing` 원문 전체 문자열이 by_type 단종 원문 집합에 있으면 참으로 표시한다.
- `사냥 시 소환되는 엘리트 몬스터 수 증가`는 같은 `reward_raw` 안에서 `(1마리→2마리)`는 단종, `(1마리→3마리)`는 현행으로 갈리는 정상 혼재 사례다.


In [4]:
tidy_rewards = tidy_prenorm.copy()

def apply_normalize(value):
    if pd.isna(value):
        return None
    return normalize_reward(str(value))

tidy_rewards['reward_norm'] = tidy_rewards['reward_raw'].apply(apply_normalize)
tidy_rewards['category'] = tidy_rewards['reward_norm'].apply(lambda x: categorize(x) if pd.notna(x) else None)
tidy_rewards['reward_magnitude'] = tidy_rewards['reward_raw'].apply(lambda x: extract_magnitude(str(x)) if pd.notna(x) else None)
tidy_rewards['cycle_days'] = compute_cycle_days(tidy_rewards)
tidy_rewards['is_discontinued'] = merge_is_discontinued(tidy_rewards, parsed_by_type)
tidy_rewards = tidy_rewards[[
    'date', 'reward_raw', 'reward_trailing', 'reward_norm', 'category', 'reward_magnitude',
    'event_context', 'is_major_event', 'is_special', 'time_limited', 'is_discontinued',
    'is_future', 'no_event_week', 'cycle_days'
]]


In [5]:
parsed_by_type.to_csv(interim_dir / 'parsed_by_type.csv', index=False, encoding='utf-8')
parsed_by_date.to_csv(interim_dir / 'parsed_by_date.csv', index=False, encoding='utf-8')
tidy_prenorm.to_csv(interim_dir / 'tidy_prenorm.csv', index=False, encoding='utf-8')
tidy_rewards.to_csv(processed_dir / 'tidy_rewards.csv', index=False, encoding='utf-8')

for name, base in [
    ('parsed_by_type.csv', interim_dir),
    ('parsed_by_date.csv', interim_dir),
    ('tidy_prenorm.csv', interim_dir),
    ('tidy_rewards.csv', processed_dir),
]:
    path = base / name
    print(f'{name}: 존재={path.exists()} 바이트={path.stat().st_size}')


parsed_by_type.csv: 존재=True 바이트=84712
parsed_by_date.csv: 존재=True 바이트=110137
tidy_prenorm.csv: 존재=True 바이트=125675
tidy_rewards.csv: 존재=True 바이트=176887


In [6]:
print('parsed_by_type 행 수:', len(parsed_by_type))
print('parsed_by_date 행 수:', len(parsed_by_date))
print('tidy_prenorm 행 수:', len(tidy_prenorm))
print('tidy_prenorm 고유 날짜 수:', tidy_prenorm['date'].nunique())
print('tidy_prenorm 최소 날짜:', tidy_prenorm['date'].min())
print('tidy_prenorm 최대 날짜:', tidy_prenorm['date'].max())
print('미래 행 수:', int(tidy_prenorm['is_future'].sum()))
print('미래 최소 날짜:', tidy_prenorm.loc[tidy_prenorm['is_future'], 'date'].min())
print('no_event_week 행 수:', int(tidy_prenorm['no_event_week'].sum()))
print('대형 이벤트 비율:', float(tidy_prenorm['is_major_event'].mean()))
print('parsed_by_type 단종 고유 보상 수:', parsed_by_type.loc[parsed_by_type['is_discontinued'], 'reward_raw'].nunique())


parsed_by_type 행 수: 1140
parsed_by_date 행 수: 1160
tidy_prenorm 행 수: 1160
tidy_prenorm 고유 날짜 수: 501
tidy_prenorm 최소 날짜: 2017-03-12
tidy_prenorm 최대 날짜: 2026-08-30
미래 행 수: 12
미래 최소 날짜: 2026-06-14
no_event_week 행 수: 30
대형 이벤트 비율: 0.7293103448275862
parsed_by_type 단종 고유 보상 수: 71


In [7]:
missing_norm = tidy_rewards['reward_norm'].isna()
unmapped = tidy_rewards[missing_norm & tidy_rewards['reward_raw'].notna()]
category_counts = tidy_rewards['category'].value_counts(dropna=False)
scored = tidy_rewards[tidy_rewards['reward_norm'].notna()]
date_raw_duplicate_count = int(tidy_rewards.loc[tidy_rewards['reward_raw'].notna()].duplicated(['date', 'reward_raw']).sum())
duplicate_mask = scored.duplicated(['date', 'reward_norm'], keep=False)
duplicate_pairs = (
    scored.loc[duplicate_mask, ['date', 'reward_norm']]
    .drop_duplicates()
    .sort_values(['date', 'reward_norm'])
)
bundled_top_norms = (
    scored.loc[duplicate_mask, ['date', 'reward_norm']]
    .drop_duplicates()
    .groupby('reward_norm')
    .size()
    .sort_values(ascending=False)
    .head(10)
)
future_empty = tidy_rewards[tidy_rewards['is_future'] & tidy_rewards['reward_raw'].isna()]
no_event_count = int(tidy_rewards['no_event_week'].sum())
future_empty_count = len(future_empty)
negative_cycle_count = int((tidy_rewards['cycle_days'].dropna() < 0).sum())
large_cycle = tidy_rewards[tidy_rewards['cycle_days'].fillna(0) > 1500][['date', 'reward_raw', 'reward_norm', 'cycle_days']]

by_type_discontinued_raw = set(parsed_by_type.loc[parsed_by_type['is_discontinued'], 'reward_raw'].dropna().unique())
tidy_full = tidy_rewards['reward_raw'].fillna('') + tidy_rewards['reward_trailing'].fillna('')
body_match = tidy_rewards['reward_raw'].isin(by_type_discontinued_raw)
full_match = tidy_full.isin(by_type_discontinued_raw)
matched_disc_keys = set(tidy_rewards.loc[body_match, 'reward_raw'].dropna().unique()) | set(tidy_full[full_match].dropna().unique())
true_combinations = tidy_rewards.loc[tidy_rewards['is_discontinued'] == True, ['reward_raw', 'reward_trailing']].drop_duplicates()
true_combination_count = len(true_combinations)
missing_disc_keys = sorted(by_type_discontinued_raw - matched_disc_keys)
extra_disc_keys = sorted(matched_disc_keys - by_type_discontinued_raw)
mixed_raw = (
    tidy_rewards.loc[tidy_rewards['reward_raw'].notna()]
    .groupby('reward_raw')['is_discontinued']
    .nunique(dropna=True)
)
mixed_raw = mixed_raw[mixed_raw > 1]
mixed_raw_values = sorted(mixed_raw.index.tolist())
expected_mixed_raw = ['사냥 시 소환되는 엘리트 몬스터 수 증가']
unexpected_mixed_raw = sorted(set(mixed_raw_values) - set(expected_mixed_raw))
missing_expected_mixed_raw = sorted(set(expected_mixed_raw) - set(mixed_raw_values))
is_discontinued_counts = tidy_rewards['is_discontinued'].value_counts(dropna=False)

powerup = tidy_rewards[
    (tidy_rewards['date'] >= '2023-01-12')
    & (tidy_rewards['date'] <= '2023-01-18')
    & (tidy_rewards['event_context'].fillna('').str.contains('\uD30C\uC6CC\uC5C5 \uC704\uD06C'))
].copy()
dates = pd.to_datetime(tidy_rewards['date'])
non_sunday_dates = sorted(tidy_rewards.loc[dates.dt.weekday != 6, 'date'].drop_duplicates().tolist())

# by_type의 명시 주기와 by_date raw 기준 계산 주기를 비교한다. 결과는 저장하지 않고 출력만 한다.
raw_tidy = tidy_rewards.loc[tidy_rewards['reward_raw'].notna(), ['reward_raw', 'date']].drop_duplicates().copy()
raw_tidy['date_dt'] = pd.to_datetime(raw_tidy['date'])
raw_tidy = raw_tidy.sort_values(['reward_raw', 'date_dt'])
raw_tidy['computed_raw_cycle_days'] = raw_tidy.groupby('reward_raw')['date_dt'].diff().dt.days.astype('Int64')
raw_cycle_check = parsed_by_type[['reward_raw', 'date', 'stated_cycle_days']].merge(
    raw_tidy[['reward_raw', 'date', 'computed_raw_cycle_days']],
    on=['reward_raw', 'date'],
    how='left',
)
raw_cycle_mismatches = raw_cycle_check[
    raw_cycle_check['stated_cycle_days'].notna()
    & raw_cycle_check['computed_raw_cycle_days'].notna()
    & (raw_cycle_check['stated_cycle_days'].astype('Int64') != raw_cycle_check['computed_raw_cycle_days'])
]
raw_cycle_missing = raw_cycle_check[
    raw_cycle_check['stated_cycle_days'].notna()
    & raw_cycle_check['computed_raw_cycle_days'].isna()
]

assert date_raw_duplicate_count == 0
assert int(missing_norm.sum()) == no_event_count + future_empty_count
assert len(unmapped) == 0
assert int((tidy_rewards['category'] == '\uC2A4\uD398\uC15C').sum()) == 0
assert int((tidy_rewards['category'] == '\uBBF8\uBD84\uB958').sum()) == 0
assert negative_cycle_count == 0
assert len(by_type_discontinued_raw) == 71
assert len(matched_disc_keys) == 71
assert len(missing_disc_keys) == 0
assert len(extra_disc_keys) == 0
assert unexpected_mixed_raw == []
assert missing_expected_mixed_raw == []

print('전체 행 수:', len(tidy_rewards))
print('date와 reward_raw 중복 행 수:', date_raw_duplicate_count)
print('reward_norm 결측 행 수:', int(missing_norm.sum()))
print('no_event_week 행 수:', no_event_count)
print('미래 빈 블록 행 수:', future_empty_count)
print('미매핑 reward_raw 행 수:', len(unmapped))
print('카테고리 분포:')
print(category_counts)
print('스페셜 카테고리 행 수:', int((tidy_rewards['category'] == '\uC2A4\uD398\uC15C').sum()))
print('미분류 카테고리 행 수:', int((tidy_rewards['category'] == '\uBBF8\uBD84\uB958').sum()))
print('date와 reward_norm 중복 조합 수:', len(duplicate_pairs))
print('date와 reward_norm 중복 행 수:', int(duplicate_mask.sum()))
print('상위 묶음 reward_norm 조합 수:')
print(bundled_top_norms.to_string())
print('cycle_days 비결측 행 수:', int(tidy_rewards['cycle_days'].notna().sum()))
print('cycle_days 음수 행 수:', negative_cycle_count)
print('cycle_days 1500일 초과 행 수:', len(large_cycle))
if len(large_cycle):
    print(large_cycle.to_string(index=False))
print('is_discontinued 분포:')
print(is_discontinued_counts.to_string())
print('by_type 단종 원문 수:', len(by_type_discontinued_raw))
print('매칭된 단종 원문 수:', len(matched_disc_keys))
print('단종 True reward_raw와 reward_trailing 조합 수:', true_combination_count)
print('단종 원문 매칭 누락 목록:', missing_disc_keys)
print('단종 원문 초과 매칭 목록:', extra_disc_keys)
print('동일 reward_raw 안에서 단종 상태가 혼재한 목록:', mixed_raw_values)
print('예상 밖 혼재 reward_raw 목록:', unexpected_mixed_raw)
print('누락된 예상 혼재 reward_raw 목록:', missing_expected_mixed_raw)
print('raw 주기 불일치 행 수:', len(raw_cycle_mismatches))
if len(raw_cycle_mismatches):
    print(raw_cycle_mismatches.head(20).to_string(index=False))
print('raw 주기 계산 누락 행 수:', len(raw_cycle_missing))
if len(raw_cycle_missing):
    print(raw_cycle_missing.head(20).to_string(index=False))
print('고유 reward_norm 수:', tidy_rewards['reward_norm'].nunique(dropna=True))
print('파워업 위크 행 수:', len(powerup))
print(powerup[['date', 'reward_raw', 'reward_trailing', 'reward_norm', 'category', 'is_special', 'is_discontinued', 'cycle_days']].to_string(index=False))
print('최소 날짜:', tidy_rewards['date'].min())
print('최대 날짜:', tidy_rewards['date'].max())
print('일요일이 아닌 날짜 수:', len(non_sunday_dates))
print('일요일이 아닌 날짜 목록:', non_sunday_dates)


전체 행 수: 1160
date와 reward_raw 중복 행 수: 0
reward_norm 결측 행 수: 42
no_event_week 행 수: 30
미래 빈 블록 행 수: 12
미매핑 reward_raw 행 수: 0
카테고리 분포:
category
강화      357
성장      337
사냥효율    234
수집기타    190
NaN      42
Name: count, dtype: int64
스페셜 카테고리 행 수: 0
미분류 카테고리 행 수: 0
date와 reward_norm 중복 조합 수: 56
date와 reward_norm 중복 행 수: 117
상위 묶음 reward_norm 조합 수:
reward_norm
몬스터컬렉션       38
돌발미션          7
룬_경험치버프       2
버닝            2
솔에르다_접속지급     2
우르스           2
몬스터파크_경험치     1
선물_기타지급       1
파티퀘스트         1
cycle_days 비결측 행 수: 1063
cycle_days 음수 행 수: 0
cycle_days 1500일 초과 행 수: 2
      date             reward_raw reward_norm  cycle_days
2023-07-09           테라 버닝 부스터 지급          버닝        2030
2024-12-15 사냥 시 소환되는 엘리트 몬스터 수 증가   엘리트몬스터_증가        1911
is_discontinued 분포:
is_discontinued
False    807
True     311
<NA>      42
by_type 단종 원문 수: 71
매칭된 단종 원문 수: 71
단종 True reward_raw와 reward_trailing 조합 수: 72
단종 원문 매칭 누락 목록: []
단종 원문 초과 매칭 목록: []
동일 reward_raw 안에서 단종 상태가 혼재한 목록: ['사냥 시 소환되는 엘리트 몬스터 수 증가']


In [8]:
date_header_count = sum(1 for line in by_date_text.splitlines() if parse_date_header(line.strip()) is not None)
date_pattern_count = len(re.findall(r'\d{4}\.\d{2}\.\d{2}', by_date_text))
s_none_count = by_date_text.count('[[S]]아예 없음[[/S]]')
b_none_count = by_date_text.count('[[B]]없음[[/B]]')
multi_trailing_count = int(tidy_prenorm['reward_trailing'].fillna('').str.contains(r'\)\(', regex=True).sum())

print('by_date 날짜 헤더 수:', date_header_count)
print('by_date 날짜 패턴 수:', date_pattern_count)
print('고유 날짜 수와 파싱 날짜 수 일치 여부:', tidy_prenorm['date'].nunique() == parsed_by_date['date'].nunique())
print('취소선 아예 없음 표기 수:', s_none_count)
print('굵게 없음 표기 수:', b_none_count)
print('괄호 접미가 여러 개인 행 수:', multi_trailing_count)
print('미래 행이 모두 기준일 이후인지:', bool((pd.to_datetime(tidy_prenorm.loc[tidy_prenorm['is_future'], 'date']) > TODAY).all()))


by_date 날짜 헤더 수: 502
by_date 날짜 패턴 수: 503
고유 날짜 수와 파싱 날짜 수 일치 여부: True
취소선 아예 없음 표기 수: 1
굵게 없음 표기 수: 29
괄호 접미가 여러 개인 행 수: 1
미래 행이 모두 기준일 이후인지: True


In [9]:
display(parsed_by_type.head())
display(parsed_by_date.head())
display(tidy_prenorm.head())

,reward_raw,date,stated_cycle_days,is_discontinued,trailing
0,"[HEXA 스탯] 메인 스탯이 5레벨 이상일 때, 메인 스탯의 강화 확률 20% 증가",2024-07-21,NaN,False,NaN
1,"[HEXA 스탯] 메인 스탯이 5레벨 이상일 때, 메인 스탯의 강화 확률 20% 증가",2024-10-13,84.0,False,NaN
2,"[HEXA 스탯] 메인 스탯이 5레벨 이상일 때, 메인 스탯의 강화 확률 20% 증가",2025-01-12,91.0,False,NaN
3,"[HEXA 스탯] 메인 스탯이 5레벨 이상일 때, 메인 스탯의 강화 확률 20% 증가",2025-04-06,84.0,False,NaN
4,"[HEXA 스탯] 메인 스탯이 5레벨 이상일 때, 메인 스탯의 강화 확률 20% 증가",2025-07-06,91.0,False,NaN


,date,event_context,is_major_event,no_event_week,reward_raw,reward_trailing
0,2017-03-12,NaN,False,False,주문의 흔적 강화 비용 30% 할인,NaN
1,2017-03-12,NaN,False,False,"(리부트, 리부트2월드 : 하루 종일 드롭률 2배)",NaN
2,2017-03-12,NaN,False,False,돌발미션 일일 클리어 가능 횟수 2배,(3회→6회)
3,2017-03-12,NaN,False,False,돌발미션 클리어 경험치 2배,NaN
4,2017-03-19,NaN,False,True,NaN,NaN


,date,reward_raw,reward_trailing,event_context,is_major_event,is_special,time_limited,is_future,no_event_week
0,2017-03-12,주문의 흔적 강화 비용 30% 할인,NaN,NaN,False,False,NaN,False,False
1,2017-03-12,"(리부트, 리부트2월드 : 하루 종일 드롭률 2배)",NaN,NaN,False,False,NaN,False,False
2,2017-03-12,돌발미션 일일 클리어 가능 횟수 2배,(3회→6회),NaN,False,False,NaN,False,False
3,2017-03-12,돌발미션 클리어 경험치 2배,NaN,NaN,False,False,NaN,False,False
4,2017-03-19,NaN,NaN,NaN,False,False,NaN,False,True


## 저장 결과 요약

- `tidy_prenorm.csv`는 날짜별 파싱 결과에 스페셜 여부, 시간 제한 여부, 미래 여부, no-event 여부를 더한 중간 산출물이다.
- `tidy_rewards.csv`는 정규화 보상명, 카테고리, 강도, 주기, 단종 여부를 포함한 최종 tidy 산출물이다.
- 명시적으로 보상이 없는 주차는 `no_event_week=True`, 보상 미공개 미래 빈 블록은 `is_future=True`로 구분한다.
